# Possession Signal SPC - Event Shift Analysis

This notebook checks whether goals, cards, and penalties are followed by possession shifts in the selected fixtures. The analysis uses a smaller shift threshold than a full 3-sigma control chart because the goal is to detect interpretable event-related movement, not formal process-control alarms.

In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("../data/ie423_match_data.csv")
SELECTED_FIXTURES = [19155147, 19155103, 19172041, 19155178, 19155143, 19172038, 19172095, 19155166]
EVENT_COLUMNS = [
    "Goals_home", "Goals_away",
    "Yellowcards_home", "Yellowcards_away",
    "Redcards_home", "Redcards_away",
    "Penalties_home", "Penalties_away",
]

def load_match_data(path=DATA_PATH):
    df = pd.read_csv(path, encoding="ISO-8859-1")
    df = df.sort_values(["fixture_id", "minute"]).reset_index(drop=True)
    required = {
        "fixture_id", "name", "minute", "Ball.Possession_home", "Ball.Possession_away",
        "Successful.Passes_home", "Successful.Passes_away",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return df

def control_limits(series, z=3.0):
    values = pd.Series(series).dropna().astype(float)
    center = values.mean()
    sigma = values.std(ddof=0)
    return center, center + z * sigma, center - z * sigma

def event_minutes(match_df):
    rows = []
    previous = match_df[EVENT_COLUMNS].shift(fill_value=0)
    event_deltas = match_df[EVENT_COLUMNS] - previous
    for idx, row in match_df.iterrows():
        for col in EVENT_COLUMNS:
            if event_deltas.loc[idx, col] > 0:
                rows.append({"minute": int(row["minute"]), "event": col.replace("_", " ")})
    return pd.DataFrame(rows)

def display_dataset_summary(df):
    summary = pd.DataFrame({
        "metric": ["rows", "fixtures", "first_minute", "last_minute"],
        "value": [len(df), df["fixture_id"].nunique(), int(df["minute"].min()), int(df["minute"].max())],
    })
    display(summary)
    display(df.head(8))


df = load_match_data()
display_dataset_summary(df)

,metric,value
0,rows,9863
1,fixtures,102
2,first_minute,1
3,last_minute,101


,fixture_id,name,halftime,minute,Successful.Passes_home,Successful.Passes_away,Ball.Possession_home,Ball.Possession_away,Goals_home,Goals_away,Attacks_home,Attacks_away,Yellowcards_home,Yellowcards_away,Redcards_home,Redcards_away,Penalties_home,Penalties_away
0,19134453,Manchester United vs Fulham,1st-half,1,0,1,0,100,0,0,0,2,0,0,0,0,0,0
1,19134453,Manchester United vs Fulham,1st-half,2,1,3,40,60,0,0,0,5,0,0,0,0,0,0
2,19134453,Manchester United vs Fulham,1st-half,3,20,5,75,25,0,0,2,5,0,0,0,0,0,0
3,19134453,Manchester United vs Fulham,1st-half,4,32,5,83,17,0,0,4,6,0,0,0,0,0,0
4,19134453,Manchester United vs Fulham,1st-half,5,32,5,83,17,0,0,4,10,0,0,0,0,0,0
5,19134453,Manchester United vs Fulham,1st-half,6,32,24,55,45,0,0,4,10,0,0,0,0,0,0
6,19134453,Manchester United vs Fulham,1st-half,7,32,24,55,45,0,0,5,10,0,0,0,0,0,0
7,19134453,Manchester United vs Fulham,1st-half,8,32,24,55,45,0,0,5,12,0,0,0,0,0,0


## Detecting post-event shifts

For each event, the notebook compares possession before and after the event in a configurable time window. The output records the direction and magnitude of the possession change. Positive values favor the home side; negative values favor the away side.

In [2]:

def detect_event_shifts(match_df, pre_window=5, post_window=10, min_abs_shift_pp=5.0):
    records = []
    events = event_minutes(match_df)
    for _, event in events.iterrows():
        minute = int(event["minute"])
        pre = match_df[(match_df["minute"] >= minute - pre_window) & (match_df["minute"] < minute)]
        post = match_df[(match_df["minute"] > minute) & (match_df["minute"] <= minute + post_window)]
        if pre.empty or post.empty:
            continue
        pre_home = pre["Ball.Possession_home"].mean()
        post_home = post["Ball.Possession_home"].mean()
        shift = post_home - pre_home
        records.append({
            "fixture_id": int(match_df["fixture_id"].iloc[0]),
            "match": match_df["name"].iloc[0],
            "event_minute": minute,
            "event": event["event"],
            "pre_home_possession": round(pre_home, 2),
            "post_home_possession": round(post_home, 2),
            "home_shift_pp": round(shift, 2),
            "shift_flag": abs(shift) >= min_abs_shift_pp,
            "direction": "home_gain" if shift > 0 else "away_gain",
        })
    return records

all_records = []
for fixture_id in SELECTED_FIXTURES:
    match_df = df[df["fixture_id"] == fixture_id].copy()
    all_records.extend(detect_event_shifts(match_df))

event_shift_table = pd.DataFrame(all_records)
display(event_shift_table.sort_values(["fixture_id", "event_minute"]))


,fixture_id,match,event_minute,event,pre_home_possession,post_home_possession,home_shift_pp,shift_flag,direction
8,19155103,Milan vs Venezia,2,Goals home,14.0,29.8,15.8,True,home_gain
9,19155103,Milan vs Venezia,16,Goals home,37.8,48.4,10.6,True,home_gain
10,19155103,Milan vs Venezia,22,Penalties home,46.0,49.6,3.6,False,home_gain
11,19155103,Milan vs Venezia,25,Goals home,50.6,49.9,-0.7,False,away_gain
12,19155103,Milan vs Venezia,29,Goals home,49.6,50.4,0.8,False,home_gain
...,...,...,...,...,...,...,...,...,...
71,19172095,Adana Demirspor vs Sivasspor,39,Yellowcards home,59.4,59.7,0.3,False,home_gain
72,19172095,Adana Demirspor vs Sivasspor,65,Goals away,53.6,54.3,0.7,False,home_gain
73,19172095,Adana Demirspor vs Sivasspor,69,Goals away,53.6,54.9,1.3,False,home_gain
74,19172095,Adana Demirspor vs Sivasspor,92,Goals away,55.0,55.5,0.5,False,home_gain


## Summary by event type

This aggregate view shows which event types were most often followed by possession changes of at least five percentage points within the next ten minutes.

In [3]:

if event_shift_table.empty:
    print("No event windows were available for the selected fixtures.")
else:
    summary = (
        event_shift_table
        .assign(event_type=lambda d: d["event"].str.replace(" home", "").str.replace(" away", ""))
        .groupby("event_type")
        .agg(
            events=("event", "count"),
            flagged_shifts=("shift_flag", "sum"),
            avg_abs_shift_pp=("home_shift_pp", lambda s: round(s.abs().mean(), 2)),
            max_abs_shift_pp=("home_shift_pp", lambda s: round(s.abs().max(), 2)),
        )
        .reset_index()
        .sort_values(["flagged_shifts", "avg_abs_shift_pp"], ascending=False)
    )
    display(summary)
    display(event_shift_table[event_shift_table["shift_flag"]].sort_values("home_shift_pp", key=lambda s: s.abs(), ascending=False))


,event_type,events,flagged_shifts,avg_abs_shift_pp,max_abs_shift_pp
0,Goals,50,7,2.67,19.2
3,Yellowcards,27,2,1.74,7.6
2,Redcards,4,0,1.00,2.3
1,Penalties,7,0,0.94,3.6


,fixture_id,match,event_minute,event,pre_home_possession,post_home_possession,home_shift_pp,shift_flag,direction
0,19155147,Atalanta vs Hellas Verona,6,Goals home,56.0,75.2,19.2,True,home_gain
8,19155103,Milan vs Venezia,2,Goals home,14.0,29.8,15.8,True,home_gain
9,19155103,Milan vs Venezia,16,Goals home,37.8,48.4,10.6,True,home_gain
20,19172041,Adana Demirspor vs Galatasaray,10,Goals away,67.8,58.2,-9.6,True,away_gain
21,19172041,Adana Demirspor vs Galatasaray,18,Goals away,60.0,50.7,-9.3,True,away_gain
67,19172095,Adana Demirspor vs Sivasspor,16,Goals home,79.0,69.9,-9.1,True,away_gain
68,19172095,Adana Demirspor vs Sivasspor,32,Yellowcards away,67.2,59.6,-7.6,True,away_gain
22,19172041,Adana Demirspor vs Galatasaray,19,Yellowcards home,57.4,50.4,-7.0,True,away_gain
56,19172038,_stanbul Ba_ak_ehir vs Antalyaspor,2,Goals away,55.0,61.7,6.7,True,home_gain


## Interpretation notes

- A flagged shift means the post-event possession average moved by at least five percentage points compared with the pre-event window.
- The method is descriptive. It is useful for report interpretation, but it does not prove that the event caused the possession shift.
- Goals and cards should be reviewed with the control charts because tactical reactions can create delayed effects.